In [ ]:
import numpy as np
import os
import torch
import torch.nn as nn
import time
import pandas as pd
import json
import logging
import ast
from tqdm.auto import tqdm
from scipy.stats import pearsonr
from collections import deque
from torch.utils.data import Dataset
from model2Vec.util import Normalizer
from model2Vec.database_util import collator, ModelGraphEncoder, WeisfeilerLehmanEncoder
from model2Vec.dataset import ModelGraphTreeNode, ModelComputationGraphDataset
from model2Vec.model import Model2Vec

In [ ]:
class Args:
    # bs = 1024
    # SQ: smaller batch size
    bs = 128
    lr = 0.001
    # epochs = 200
    epochs = 20
    clip_size = 50
    embed_size = 64
    pred_hid = 128
    ffn_dim = 128
    head_size = 12
    n_layers = 8
    dropout = 0.1
    sch_decay = 0.6
    device = 'cuda:0'
    newpath = './results/full/cost/'
    to_predict = 'cost'
  
args = Args()
if not os.path.exists(args.newpath):
    os.makedirs(args.newpath)

In [ ]:
from model2Vec.util import seed_everything
seed_everything()

In [ ]:
to_predict = 'cost'
to_predict = 'cost'
dir_path = "/home/velox/velox/optimizer/tests"
df2 = pd.read_csv(os.path.join(dir_path, "generatedQueryPlan", "query_benchmark_results_YOURTIMESTAMP.csv"), sep="|")
df = df2
df = df[df["error"].isna()]
max_latency = np.max(df["executionTime"])
cost_norm = Normalizer(0, max_latency)
model_graph_encoder = ModelGraphEncoder()

In [ ]:
from sklearn.model_selection import train_test_split

# Split the dataframe into 80% training and 20% testing
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_ds = ModelComputationGraphDataset(cost_normalizer=cost_norm, model_graph_encoder=model_graph_encoder, df=train_df, to_predict="cost")
test_ds = ModelComputationGraphDataset(cost_normalizer=cost_norm, model_graph_encoder=model_graph_encoder, df=test_df, to_predict="cost")

In [ ]:
from model2Vec.trainer import train_model2vec, train_model2vec_with_contrastive
from model2Vec import trainer
args.bs = 32
args.device = "cuda:0"
model = Model2Vec(emb_size = args.embed_size ,ffn_dim = args.ffn_dim, head_size = args.head_size, \
                 dropout = args.dropout, n_layers = args.n_layers)
_ = model.to(args.device)
crit = nn.MSELoss()

In [ ]:
wl_encoder = WeisfeilerLehmanEncoder(range_percent=0.2, num_iterations=2)
wl_encoder.obtain_wl_feature_for_dataset(train_ds)
wl_encoder.construct_similar_dissimilar_pairs_for_dataset(train_ds)

In [ ]:
model = Model2Vec(emb_size = args.embed_size ,ffn_dim = args.ffn_dim, head_size = args.head_size, \
                 dropout = args.dropout, n_layers = args.n_layers)
_ = model.to(args.device)
args.epochs = 20
model, best_path = train_model2vec_with_contrastive(model, train_ds, train_ds, crit, cost_norm, args, contrastive_loss_factor=1, cost_loss_factor=0)

In [ ]:
model = Model2Vec(emb_size = args.embed_size ,ffn_dim = args.ffn_dim, head_size = args.head_size, \
                 dropout = args.dropout, n_layers = args.n_layers)
_ = model.to(args.device)
args.epochs = 20
model, best_path = train_model2vec_with_contrastive(model, train_ds, train_ds, crit, cost_norm, args, contrastive_loss_factor=0, cost_loss_factor=1)

In [ ]:
# save pre-trained model
torch.save(model, './cactusdb/model2vec_finetune/model2vec.pt')
torch.save(cost_norm, './cactusdb/model2vec_finetune/cost_norm.pt')
torch.save(wl_encoder, './cactusdb/model2vec_finetune/wl_encoder.pt')
torch.save(model_graph_encoder, './cactusdb/model2vec_finetune/model_graph_encoder.pt')